# MART_P09_HUBS_HOSPITALARES

**P09. Quais hospitais funcionam como "hubs", recebendo pacientes do maior número de
municípios?** *(grafo)*

Cada linha é uma **aresta** município de residência → hospital, com o volume de
internações e o valor. No grafo, o grau de entrada de cada hospital (municípios
distintos ligados a ele) mede sua centralidade na rede.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P09_HUBS_HOSPITALARES
    (COD_IBGE_ORIGEM, MUNICIPIO_ORIGEM, COD_CNES, NOME_ESTABELECIMENTO, MUNICIPIO_HOSPITAL,
     QTD_INTERNACOES, VL_TOTAL)
SELECT m.COD_IBGE_6, m.NOME_MUNICIPIO, e.COD_CNES, e.NOME_ESTABELECIMENTO, e.NOME_MUNICIPIO,
       SUM(f.QTD_INTERNACAO), SUM(f.VL_TOTAL)
FROM FATO_INTERNACAO f
JOIN DIM_MUNICIPIO m       ON m.SK_MUNICIPIO = f.SK_MUNICIPIO_RESIDENCIA
JOIN DIM_ESTABELECIMENTO e ON e.SK_ESTABELECIMENTO = f.SK_ESTABELECIMENTO
WHERE f.SK_MUNICIPIO_RESIDENCIA <> -1
  AND f.SK_ESTABELECIMENTO <> -1
GROUP BY m.COD_IBGE_6, m.NOME_MUNICIPIO, e.COD_CNES, e.NOME_ESTABELECIMENTO, e.NOME_MUNICIPIO
HAVING SUM(f.QTD_INTERNACAO) > 0
"""

In [ ]:
recarregar("MART_P09_HUBS_HOSPITALARES", SQL, origem="FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM MART_P09_HUBS_HOSPITALARES FETCH FIRST 10 ROWS ONLY")